# Retrieval-Augmented Generation (RAG) for Computational Social Science

---

<div class="alert alert-success">  
    
### Learning Objectives 

* Understand how RAG combines retrieval and generation for better LLM outputs
* Learn to build a simple RAG system for social science research
* Apply RAG to content labeling and classification tasks
* Evaluate when RAG is more appropriate than standard LLM prompting
* Develop skills to work with document collections in research projects
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>

### Sections
1. [What is RAG and Why Does It Matter?](#section1)
2. [Setting Up Our RAG System](#section2)
3. [Building a Simple RAG Pipeline](#section3)
4. [RAG for Social Science: Labeling and Classification](#section4)
5. [Example: Analyzing Policy Documents](#section5)
6. [Evaluating RAG vs Standard Prompting](#section6)

**Why this matters: RAG allows you to ground LLM outputs in your specific documents and data, making them more accurate and reliable for research tasks like content analysis, coding qualitative data, and literature review.**

<a id='section1'></a>

## What is RAG and Why Does It Matter?

### The Challenge

Large Language Models (LLMs) are powerful, but they have limitations:
- They can hallucinate or make up information
- They don't know about your specific documents or data
- Their knowledge cutoff means they miss recent information
- They can't reason over large document collections

### Enter RAG

**Retrieval-Augmented Generation** solves this by combining two steps:

1. **Retrieval**: Find relevant documents from your collection
2. **Generation**: Use those documents as context for the LLM

<img src='../../img/rag.png' alt="RAG Process" width="600">

Think of it like an open-book exam vs. a closed-book exam. RAG lets the LLM consult your specific materials before answering.

### RAG in Computational Social Science

RAG is particularly valuable for:

**Content Labeling & Classification**
- Label social media posts with reference to a codebook
- Classify documents using examples from your corpus
- Code interview transcripts with theoretical frameworks

**Literature Review & Synthesis**
- Search across hundreds of papers
- Identify relevant citations and connections
- Summarize findings from specific subfields

**Policy Analysis**
- Compare policy documents across jurisdictions
- Track how specific issues are addressed
- Identify patterns in legislative language

**Qualitative Data Analysis**
- Ground LLM interpretations in your interview data
- Retrieve similar examples when coding
- Maintain consistency across large qualitative datasets

### Real-World Example: Community Response Forecasting

A recent application of RAG in social computing research demonstrates its power:

**SCRAG** (Sun et al., 2025) - [Social Computing-Based RAG](https://arxiv.org/abs/2504.16947)

This research forecasts how online communities will respond to new posts by retrieving:
1. **Historical responses** from the target community to capture their ideological, semantic, and emotional patterns
2. **External knowledge** from news articles to provide time-sensitive context

The system combines these retrieved elements to predict community reactions, demonstrating how RAG can capture both community-specific norms and broader contextual information.

💡 **Tip**: RAG is especially useful when you need the LLM to be faithful to specific source material rather than generating creative but potentially inaccurate content.

<a id='section2'></a>

## Setting Up Our RAG System

We'll build a simple but functional RAG system using:
- **sentence-transformers**: For creating embeddings
- **FAISS**: For fast similarity search
- **Google's Gemini**: As our LLM (free tier available)

⚠️ **Warning**: You'll need a Gemini API key. Get one free at https://aistudio.google.com/app/apikey

In [1]:
# Install required packages (run once)
#!pip install sentence-transformers faiss-cpu google-genai -q

In [2]:
# Import basic libraries
import numpy as np
import os

In [3]:
# Import FAISS for vector similarity search
import faiss

In [4]:
# Import sentence transformers for creating embeddings
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

/Users/tomvannuenen/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/tomvannuenen/anaconda3/envs/llm/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Embedding dimension: 384


In [5]:
# Import and configure Gemini for text generation
from google import genai

# Set your API key here or use environment variable
GEMINI_API_KEY = os.getenv('GOOGLE_API_KEY', 'your-api-key-here')

# Create a client with the API key
client = genai.Client(api_key=GEMINI_API_KEY)

🔔 **Question**: Why do we need separate embedding and generation models? Why not just use the LLM for everything?

<details>
<summary>Click for answer</summary>
Embedding models are optimized for finding semantic similarity, while LLMs are optimized for text generation. Using specialized models for each task is more efficient and often more accurate.
</details>

<a id='section3'></a>

## Building a Simple RAG Pipeline

Let's create a basic RAG system with three core components:

1. **Document Store**: Holds our documents and their embeddings
2. **Retriever**: Finds relevant documents for a query
3. **Generator**: Uses retrieved docs to answer questions

In [6]:
class SimpleRAG:
    """A minimal RAG implementation for teaching purposes."""
    
    def __init__(self, embedding_model, genai_client, llm_model_name='gemini-2.0-flash'):
        """Initialize the RAG system with models."""
        self.embedding_model = embedding_model
        self.client = genai_client  # Store the genai client
        self.model_name = llm_model_name  # Store model name
        self.documents = []  # Store all documents
        self.index = None    # FAISS index for similarity search

    def add_documents(self, documents):
        """Add documents to our knowledge base."""
        # Save documents
        self.documents.extend(documents)
        
        # Convert documents to embeddings (vectors)
        embeddings = self.embedding_model.encode(documents)
        
        # Create FAISS index if it doesn't exist
        dimension = embeddings.shape[1]
        if self.index is None:
            # L2 distance metric
            self.index = faiss.IndexFlatL2(dimension)
        
        # Add embeddings to index
        self.index.add(np.array(embeddings).astype('float32'))
        
        print(f"✓ Added {len(documents)} documents. Total: {len(self.documents)}")

    def retrieve(self, query, k=3):
        """Retrieve k most relevant documents for the query."""
        # Convert query to embedding
        query_embedding = self.embedding_model.encode([query])
        
        # Search FAISS index for similar documents
        # Returns distances and indices of k nearest neighbors
        distances, indices = self.index.search(
            np.array(query_embedding).astype('float32'), k
        )
        
        # Package results with documents and their similarity scores
        results = [
            {'document': self.documents[idx], 'distance': dist}
            for idx, dist in zip(indices[0], distances[0])
        ]
        return results
    
    def generate(self, query, context_docs):
        """Generate answer using retrieved documents as context."""
        # Format retrieved documents into context string
        context = "\n\n".join([
            f"Document {i+1}: {doc['document']}" 
            for i, doc in enumerate(context_docs)
        ])
        
        # Build prompt with context + query
        prompt = f"""Based on the following documents, please answer the question.
        
                Documents:
                {context}

                Question: {query}

                Answer based on the documents provided:"""
        
        # Generate response from LLM
        # Generate response using client-based API
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )
        return response.text
    
    def query(self, question, k=3, show_context=False):
        """Complete RAG pipeline: retrieve then generate."""
        # Step 1: Retrieve relevant documents
        retrieved_docs = self.retrieve(question, k=k)
        
        # Optionally show what was retrieved
        if show_context:
            print("\n📚 Retrieved documents:")
            for i, doc in enumerate(retrieved_docs, 1):
                print(f"{i}. {doc['document'][:100]}...")
            print()
        
        # Step 2: Generate answer using retrieved context
        answer = self.generate(question, retrieved_docs)
        return answer

💡 **Tip**: Notice how we separated retrieval (finding relevant docs) from generation (using them to answer). This modularity makes it easier to improve each component independently.

In [7]:
# Create our RAG system
rag = SimpleRAG(embedding_model, client)

### Let's Test It with Sample Documents

We'll use a small collection of research abstracts about social media and misinformation.

In [8]:
# Sample research abstracts (shortened for demo)
sample_documents = [
    "A study of 126,000 Twitter users found that false news spreads faster and further than true news. False stories were 70% more likely to be retweeted. This suggests emotional content and novelty drive sharing behavior more than accuracy.",
    
    "Analysis of Facebook engagement shows that outrage and moral-emotional language receive significantly more engagement. Posts containing moral-emotional words received 20% more engagement than those without. This creates incentives for polarizing content.",
    
    "Echo chambers on social media limit exposure to diverse viewpoints. Users primarily interact with like-minded individuals, reinforcing existing beliefs. Only 13% of political conversations crossed party lines on Twitter during the 2020 election.",
]

In [9]:
# More sample documents
more_documents = [
    "Platform recommendation algorithms optimize for engagement, which often means amplifying divisive content. YouTube's recommendation system was found to suggest increasingly extreme content. A/B testing showed engagement-optimized algorithms led to more polarization.",
    
    "Fact-checking interventions have mixed effectiveness. While warning labels reduce sharing of false content by 20-30%, they can also trigger reactance among some users. Timing and framing of corrections matter significantly for effectiveness.",
    
    "Bot networks can amplify misleading narratives at scale. Research identified coordinated bot campaigns reaching millions of users. These networks often exploit trending topics and use sophisticated language to appear human.",
]

In [10]:
# Add all documents to our RAG system
rag.add_documents(sample_documents)
rag.add_documents(more_documents)

✓ Added 3 documents. Total: 3
✓ Added 3 documents. Total: 6


In [11]:
# Define our test question
question = "What factors make false information spread on social media?"

In [12]:
# Query with context shown (k=3 means retrieve 3 most relevant documents)
answer = rag.query(question, k=3, show_context=True)
print(f"\nAnswer: {answer}")


📚 Retrieved documents:
1. A study of 126,000 Twitter users found that false news spreads faster and further than true news. Fa...
2. Bot networks can amplify misleading narratives at scale. Research identified coordinated bot campaig...
3. Echo chambers on social media limit exposure to diverse viewpoints. Users primarily interact with li...


Answer: Based on the documents provided, the following factors contribute to the spread of false information on social media:

*   **Emotional content and novelty:** False news is more likely to be retweeted due to its emotional appeal and novelty compared to true news (Document 1).
*   **Bot networks:** Coordinated bot campaigns can amplify misleading narratives, reaching millions of users (Document 2).
*   **Echo chambers:** Limited exposure to diverse viewpoints within echo chambers reinforces existing beliefs, allowing false information to spread more easily among like-minded individuals (Document 3).



💡 **Tip**: Notice how the answer is grounded in the specific documents we provided. The LLM isn't making up studies - it's citing the evidence we gave it!

<a id='section6'></a>

## Evaluating RAG vs Standard Prompting

Let's compare RAG to standard prompting to understand when each is appropriate.

In [13]:
# Define a function for standard prompting (no retrieval)
def standard_prompt(question):
    """Query LLM directly without retrieval."""
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=question
    )
    return response.text

In [14]:
# Test question about our specific documents
test_question = "What percentage of political conversations crossed party lines on Twitter during the 2020 election?"

In [15]:
# Get answer without RAG
print("\n--- STANDARD PROMPTING (No RAG) ---")
print(standard_prompt(test_question))


--- STANDARD PROMPTING (No RAG) ---
Unfortunately, I don't have access to precise, definitive data on the percentage of political conversations crossing party lines on Twitter during the 2020 election. Tracking this would require sophisticated analysis of user affiliations and conversation patterns, and it's not information that is publicly available in a comprehensive way.

However, I can offer some context and potential avenues for finding related information:

*   **Research is limited but suggests echo chambers:** Studies on Twitter during the 2020 election generally point to the prevalence of "echo chambers" and partisan polarization. This means that users tend to interact more with accounts and content that align with their existing political views.

*   **Academic studies and reports:** You might find relevant data in academic studies, reports by organizations that analyze social media, or articles that specifically investigated cross-party interactions on Twitter during that p

In [16]:
# Get answer with RAG
print("\n--- WITH RAG ---")
print(rag.query(test_question, k=2, show_context=False))


--- WITH RAG ---
13%



### When to Use RAG vs Standard Prompting

**Use RAG when:**
- You need answers grounded in specific documents
- Working with specialized or domain-specific content
- Need to cite sources from your corpus
- Documents are too long to fit in context window
- Want consistency in how documents are interpreted
- Coding/labeling with a detailed codebook

**Use Standard Prompting when:**
- Task requires general knowledge
- You want creative generation
- No specific source documents needed
- Working with small amounts of context
- Speed is critical (RAG adds retrieval overhead)

⚠️ **Warning**: RAG is only as good as your document collection. If relevant information isn't in your documents, RAG can't help. "Garbage in, garbage out" applies here!

<a id='section4'></a>

## RAG for Labeling and Classification

One of the most powerful applications of RAG in computational social science is **content labeling**. Instead of hoping the LLM understands your coding scheme, you can provide it with:
- Your codebook as context
- Examples of previously coded content
- Theoretical frameworks to apply

### Example: Coding Social Media Posts with a Codebook

In [17]:
# Create a codebook for political communication research
codebook = [
    "CODE 1 - POLICY DISCUSSION: Post focuses on specific policy proposals, legislation, or government programs. Discusses implementation, effectiveness, or consequences of policies. Example: 'The new healthcare bill will expand coverage to 2 million people.'",
    
    "CODE 2 - IDENTITY POLITICS: Post emphasizes group identity, representation, or identity-based claims. Focuses on experiences of specific demographic groups. Example: 'As a working-class woman, I know what families need.'",
    
    "CODE 3 - MORAL OUTRAGE: Post expresses strong moral condemnation or outrage about issues or opponents. Uses emotional, judgmental language. Example: 'This is a disgusting attack on our values by corrupt politicians.'",
    
    "CODE 4 - HORSE RACE: Post focuses on electoral competition, polling, strategy, or political tactics rather than issues. Example: 'Latest poll shows candidate leading by 5 points in swing states.'",
    
    "CODE 5 - FACT/INFORMATION SHARING: Post shares factual information, statistics, or reports without strong opinion. Example: 'New study finds 60% of residents support the measure.'",
]

# Create a RAG system for coding
coding_rag = SimpleRAG(embedding_model, client)
coding_rag.add_documents(codebook)

✓ Added 5 documents. Total: 5


In [18]:
# Posts to code
posts_to_code = [
    "We need to stand up for REAL Americans who work hard and play by the rules. The elite politicians don't care about us!",
    "The proposed infrastructure bill allocates $50B for roads, $25B for bridges, and includes tax incentives for green energy.",
    "Senator Smith maintains a 12-point lead according to the latest Quinnipiac poll. Campaign momentum appears strong.",
]

def code_post(post, rag_system):
    """Use RAG to code a post according to our codebook."""
    prompt = f"""Code this social media post according to the codebook. 
    Return ONLY the code number (1-5) and a brief explanation.
    
    Post: {post}"""
    
    return rag_system.query(prompt, k=2, show_context=False)

# Code each post
print("\n=== CODING RESULTS ===")
for i, post in enumerate(posts_to_code, 1):
    print(f"\nPost {i}: {post}")
    print(f"Code: {code_post(post, coding_rag)}")
    print("-" * 80)


=== CODING RESULTS ===

Post 1: We need to stand up for REAL Americans who work hard and play by the rules. The elite politicians don't care about us!
Code: 1 - Moral outrage: The post uses judgmental language ("REAL Americans," "elite politicians") and expresses condemnation of politicians who "don't care about us."

--------------------------------------------------------------------------------

Post 2: The proposed infrastructure bill allocates $50B for roads, $25B for bridges, and includes tax incentives for green energy.
Code: 5 - This post shares factual information about the infrastructure bill's allocations without expressing a strong opinion.

--------------------------------------------------------------------------------

Post 3: Senator Smith maintains a 12-point lead according to the latest Quinnipiac poll. Campaign momentum appears strong.
Code: 4 - This post focuses on electoral competition and polling data, fitting the description of a horse race post.

--------------

🔔 **Question**: How does using RAG for coding differ from just prompting the LLM with your codebook? What are the advantages?

<details>
<summary>Click for answer</summary>
RAG automatically retrieves the most relevant codes based on semantic similarity, meaning the LLM only sees the most applicable parts of your codebook. This is especially valuable when you have large codebooks (20+ codes) that exceed the LLM's attention span. It also allows you to include many examples without hitting context length limits.
</details>

### Example-Based RAG: Learning from Previously Coded Content

One of the most powerful applications of RAG is **Example-RAG** (also called k-Nearest Neighbor RAG or kNN-RAG). Instead of just providing abstract code definitions, you retrieve similar *previously coded examples* to guide the LLM.

This is similar to how human coders work: when unsure about a coding decision, they look at similar cases they've already coded and apply the same logic.

**Benefits:**
- More consistent coding across large datasets
- Helps with edge cases by finding similar examples
- Reduces need for extensive codebook rules
- Captures implicit patterns in your coding decisions

In [19]:
# Previously coded examples with gold standard labels
coded_examples = [
    {"text": "Senator introduces bill to expand Medicaid coverage to include dental care for all enrollees.",
     "code": 1, "label": "POLICY DISCUSSION"},
    
    {"text": "As a teacher in this district, I see firsthand how budget cuts hurt our kids every day.",
     "code": 2, "label": "IDENTITY POLITICS"},
    
    {"text": "This is absolutely shameful! These politicians are selling out our future for corporate donors.",
     "code": 3, "label": "MORAL OUTRAGE"},
    
    {"text": "New Rasmussen poll shows Governor ahead by 8 points among likely voters in key districts.",
     "code": 4, "label": "HORSE RACE"},
    
    {"text": "Census data shows median household income increased 3.2% in the state last year.",
     "code": 5, "label": "FACT/INFORMATION SHARING"},
    
    {"text": "Proposed legislation would raise minimum teacher salary to $50k and provide retention bonuses.",
     "code": 1, "label": "POLICY DISCUSSION"},
    
    {"text": "Our community has been ignored for too long. Time for representatives who actually understand us.",
     "code": 2, "label": "IDENTITY POLITICS"},
    
    {"text": "Campaign raised $2.3M this quarter, leading all challengers in the district.",
     "code": 4, "label": "HORSE RACE"},
]

In [20]:
# Create Example-RAG system with coded examples
example_rag = SimpleRAG(embedding_model, client)

# Format examples as documents with their codes
example_docs = [
    f"Text: {ex['text']}\nCode: {ex['code']} - {ex['label']}"
    for ex in coded_examples
]

example_rag.add_documents(example_docs)

✓ Added 8 documents. Total: 8


In [21]:
def code_with_examples(post, rag_system, k=3):
    """Code a post by retrieving similar previously coded examples."""
    prompt = f"""Based on the similar examples provided, code this new post.
    Return ONLY the code number (1-5) and a brief explanation.
    
    New post to code: {post}"""
    
    return rag_system.query(prompt, k=k, show_context=True)

In [22]:
# Test on new posts
new_posts = [
    "Representative proposes tax credit for families earning under $75k to offset childcare costs.",
    "We working families know what it's like to struggle. Politicians in their ivory towers don't get it.",
]

print("\n=== EXAMPLE-BASED CODING ===")
for i, post in enumerate(new_posts, 1):
    print(f"\n{'='*80}")
    print(f"Post {i}: {post}")
    print(f"\nCode: {code_with_examples(post, example_rag, k=3)}")


=== EXAMPLE-BASED CODING ===

Post 1: Representative proposes tax credit for families earning under $75k to offset childcare costs.

📚 Retrieved documents:
1. Text: Proposed legislation would raise minimum teacher salary to $50k and provide retention bonuses....
2. Text: As a teacher in this district, I see firsthand how budget cuts hurt our kids every day.
Code: ...
3. Text: Senator introduces bill to expand Medicaid coverage to include dental care for all enrollees.
...


Code: 1 - POLICY DISCUSSION
The new post discusses a proposed policy change (tax credit) related to a specific group (families earning under $75k). This aligns with the examples in Document 1 and Document 3, which both describe proposed legislation/bills.


Post 2: We working families know what it's like to struggle. Politicians in their ivory towers don't get it.

📚 Retrieved documents:
1. Text: Our community has been ignored for too long. Time for representatives who actually understand ...
2. Text: As a teacher 

💡 **Tip**: Example-RAG works best when you have diverse, high-quality examples. Start by manually coding 50-100 examples across all categories, then use Example-RAG to scale up.

⚠️ **Warning**: Be careful not to retrieve examples from the same source or author as the item being coded - this can introduce bias!

### Scaling Up: Working with Larger Document Collections

In real research, you might have:
- 100+ interview transcripts
- 1000+ news articles
- Entire legislative sessions worth of documents

RAG handles this by:
1. Only retrieving the most relevant documents (k=3-10)
2. Using efficient vector search (FAISS can handle millions of documents)
3. Keeping context sizes manageable for the LLM

💡 **Tip**: For very large collections, consider chunking long documents into smaller segments. This improves retrieval precision by matching query to specific relevant sections rather than entire documents.

## References

1. [RAG Paper](https://arxiv.org/abs/2005.11401): Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS 2020*.
2. [SCRAG](https://arxiv.org/abs/2504.16947): Sun, Z., et al. (2025). SCRAG: Social computing-based retrieval augmented generation for community response forecasting in social media environments.
3. [Sentence Transformers](https://arxiv.org/abs/1908.10084): Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks.
4. [Dense Passage Retrieval](https://arxiv.org/abs/2004.04906): Karpukhin, V., et al. (2020). Dense passage retrieval for open-domain question answering.
5. [Benchmarking RAG](https://arxiv.org/abs/2309.01431): Jiawei Chen et al. (2023). Benchmarking Large Language Models in Retrieval-Augmented Generation.

<div class="alert alert-success">

## ❗ Key Points

* **RAG grounds LLM outputs in evidence**: By retrieving relevant documents first, you ensure answers are based on your specific sources rather than the model's general training.

* **Essential for content coding**: RAG allows you to provide codebooks, examples, and theoretical frameworks as context, making LLM-based coding more reliable and consistent.

* **Example-RAG improves consistency**: Retrieving previously coded examples (kNN-RAG) helps maintain consistency and handle edge cases by learning from similar past decisions.

* **Scales to large document collections**: Unlike putting everything in the prompt, RAG efficiently retrieves only the most relevant content, allowing you to work with thousands of documents.

* **Retrieval quality matters**: RAG is only as good as your document collection and retrieval strategy. Poor retrieval leads to poor generation.

* **Not always necessary**: Standard prompting is sufficient for general knowledge tasks. Use RAG when you need faithfulness to specific documents or when working with specialized content.

* **Evaluation is crucial**: Always validate that retrieved documents are relevant and that generated answers faithfully represent those documents. RAG systems can still hallucinate if not carefully designed.

</div>